# TradeSense · Component 03 · Is the rush_order_density jump in August to October real? (read only)

Notebook 25 showed that the median of `rush_order_density` (trades per unit volume) in 2024-08-01..10-31 is about 2.2 times the Layer 1 median (BTC 136.7 vs 62.2, ETH 8.3 vs 3.7), while the other five features stay close. This notebook answers whether this is (a) a real market change, (b) a step at a specific date, or (c) a source difference between the Tardis files and the Binance trade files. It prints a daily table (median trades per bar, median rush, volume) for 2024-05-01..10-31, flags the largest step, and compares Tardis against Binance trades on the three first-of-month days that exist in both. Read only; CPU, a few minutes.

In [1]:
import os, glob, zipfile, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
OUT = BASE/'layer1_aug_oct_v1'; OUT.mkdir(exist_ok=True)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def load(p):
    d = pd.read_parquet(p)
    if 'datetime' in d.columns: d = d.set_index('datetime')
    d.index = pd.to_datetime(d.index, utc=True); return d

Mounted at /content/drive


## 1 · Weekly table and largest step

In [2]:
for s in ['BTCUSDT', 'ETHUSDT']:
    old = load([x for x in glob.glob(str(BASE/'**'/f'{s.lower()}_l1_feature_matrix_90d.parquet'), recursive=True) if 'backup' not in x.lower()][0])
    new = load(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet')
    cols = ['trades_count','volume','rush_order_density','large_trade_count']
    a = pd.concat([old[cols], new[cols]]).sort_index(); a = a[~a.index.duplicated()]
    day = a.groupby(a.index.floor('D')).agg(trades_per_bar=('trades_count','median'), volume_per_bar=('volume','median'), rush_med=('rush_order_density','median'), large_med=('large_trade_count','median'))
    day['week'] = day.index.to_period('W').astype(str)
    wk = day.groupby('week').agg(trades_per_bar=('trades_per_bar','median'), volume_per_bar=('volume_per_bar','median'), rush_med=('rush_med','median'), large_med=('large_med','median'))
    say(f'===== {s}: weekly medians of per-bar values (Layer 1 covers 2024-05-01..07-29, new data 08-01..10-31)')
    say(wk.round(3).to_string())
    r = day.rush_med.to_numpy(); step = [(abs(np.log(r[i:i+7].mean()/r[i-7:i].mean())), day.index[i]) for i in range(7, len(r)-7)]
    best = max(step); say(f'largest 7-day step in rush_med: factor {np.exp(best[0]):.2f} at {best[1].date()}')
    say('')

===== BTCUSDT: weekly medians of per-bar values (Layer 1 covers 2024-05-01..07-29, new data 08-01..10-31)
                       trades_per_bar  volume_per_bar  rush_med  large_med
week                                                                      
2024-04-29/2024-05-05          769.00          16.555    45.998       43.5
2024-05-06/2024-05-12          688.00          12.173    59.891       30.0
2024-05-13/2024-05-19          537.00          11.322    50.379       29.0
2024-05-20/2024-05-26          686.00          13.710    49.943       36.0
2024-05-27/2024-06-02          547.00          10.289    56.858       25.0
2024-06-03/2024-06-09          519.00          11.092    49.467       25.0
2024-06-10/2024-06-16          661.00          11.924    59.816       27.0
2024-06-17/2024-06-23          622.00          10.160    64.089       24.0
2024-06-24/2024-06-30          589.50          10.744    65.719       23.0
2024-07-01/2024-07-07          793.50          10.810    72.519      

## 2 · Tardis versus Binance on the same day

In [3]:
say('===== Tardis vs Binance trade files on the same first-of-month day (same market, two sources)')
import io
rows = []
for s in ['BTCUSDT', 'ETHUSDT']:
    for d in ['2024-08-01', '2024-09-01', '2024-10-01']:
        tp = list(BASE.rglob(f'binance_trades_{d}_{s}.csv.gz'))
        if not tp: continue
        t = pd.read_csv(tp[0], usecols=['timestamp','amount'])
        zp = BASE/'layer1_rebuild_v1'/'zips'/f'{s}-trades-{d}.zip'
        try:
            urllib.request.urlretrieve(f'https://data.binance.vision/data/spot/daily/trades/{s}/{s}-trades-{d}.zip', zp)
            with zipfile.ZipFile(zp) as z, z.open(z.namelist()[0]) as fh: b = pd.read_csv(fh, header=None, usecols=[2,4])
            zp.unlink(missing_ok=True)
            tt = t[(t.timestamp >= pd.Timestamp(d, tz='UTC').value//1000)]
            rows.append((s, d, len(t), len(b), round(t.amount.sum(), 3), round(b[2].sum(), 3)))
        except Exception as e:
            rows.append((s, d, len(t), f'FAILED {type(e).__name__}', round(t.amount.sum(), 3), np.nan))
df = pd.DataFrame(rows, columns=['symbol','day','Tardis trades','Binance trades','Tardis volume','Binance volume'])
say(df.to_string(index=False))
say('Equal counts and volumes: the two sources agree, so a jump seen between Tardis days and Binance days would be a real change.')

===== Tardis vs Binance trade files on the same first-of-month day (same market, two sources)
 symbol        day  Tardis trades  Binance trades  Tardis volume  Binance volume
BTCUSDT 2024-08-01        2397256         2397257      35542.268       35542.269
BTCUSDT 2024-09-01        2153180         2153180      20705.157       20705.157
BTCUSDT 2024-10-01        5607068         5607069      43671.481       43671.481
ETHUSDT 2024-08-01        1835455         1835455     323285.610      323285.610
ETHUSDT 2024-09-01        1357443         1357443     223127.776      223127.776
ETHUSDT 2024-10-01        4525441         4525442     502828.674      502828.680
Equal counts and volumes: the two sources agree, so a jump seen between Tardis days and Binance days would be a real change.


In [4]:
(OUT/'drift_check_share_block.txt').write_text('\n'.join(LINES)); print('\nsaved', OUT/'drift_check_share_block.txt')


saved /content/drive/MyDrive/research datasets/layer1_aug_oct_v1/drift_check_share_block.txt
